In [39]:
import torch
import tiktoken

In [40]:
from torch import nn

In [47]:
torch.manual_seed(123)

In [48]:
class TokenizerV1:
    def __init__(self):
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.n_vocab = self.tokenizer.n_vocab

    def encode(self, text):
        return self.tokenizer.encode(text)

    def decode(self, encodings):
        return self.tokenizer.decode(encodings)

In [49]:
tokenizer = TokenizerV1()

In [50]:
sample_text = "I want to play football."

In [51]:
encodings = tokenizer.encode(sample_text)

In [52]:
encodings

[40, 765, 284, 711, 4346, 13]

In [53]:
# Hyperparameters
emb_dim = 4
context_length = 6

In [54]:
embeddings = nn.Embedding(tokenizer.n_vocab, emb_dim)

In [57]:
input_embeddings = embeddings(torch.tensor(encodings))

In [58]:
input_embeddings

tensor([[ 0.6851,  2.0024, -0.5469,  1.6014],
        [ 0.1603, -0.7429, -1.0349, -0.5512],
        [ 0.8393, -0.0830, -0.2559, -0.3010],
        [ 1.4175,  0.3827,  2.0482,  0.3541],
        [-0.2348,  0.0960, -0.0968, -0.9130],
        [-1.2203,  1.3139,  1.0533,  0.1388]], grad_fn=<EmbeddingBackward0>)

In [59]:
positional_embeddings = nn.Embedding(context_length, 4)

In [60]:
positional_embeddings(torch.arange(0, context_length))

tensor([[-1.3399,  0.1260,  0.2664, -0.2735],
        [ 0.7472,  0.3919,  0.3516, -2.3808],
        [-0.4979,  0.6442, -0.5104,  1.7235],
        [ 1.0708, -0.7917,  0.1897, -0.1156],
        [-0.7405, -0.1194,  0.7420, -0.3785],
        [ 0.2829, -0.9172, -0.2300, -0.7457]], grad_fn=<EmbeddingBackward0>)

In [61]:
input_embeddings_with_position = input_embeddings + positional_embeddings(torch.arange(0, context_length))

In [62]:
input_embeddings_with_position

tensor([[-0.6548,  2.1284, -0.2805,  1.3280],
        [ 0.9075, -0.3510, -0.6833, -2.9320],
        [ 0.3414,  0.5612, -0.7663,  1.4225],
        [ 2.4883, -0.4090,  2.2379,  0.2385],
        [-0.9753, -0.0234,  0.6451, -1.2915],
        [-0.9374,  0.3967,  0.8233, -0.6069]], grad_fn=<AddBackward0>)

In [63]:
inputs = input_embeddings_with_position

In [64]:
inputs.shape

torch.Size([6, 4])

In [65]:
inputs.T

tensor([[-0.6548,  0.9075,  0.3414,  2.4883, -0.9753, -0.9374],
        [ 2.1284, -0.3510,  0.5612, -0.4090, -0.0234,  0.3967],
        [-0.2805, -0.6833, -0.7663,  2.2379,  0.6451,  0.8233],
        [ 1.3280, -2.9320,  1.4225,  0.2385, -1.2915, -0.6069]],
       grad_fn=<PermuteBackward0>)

In [66]:
attention_score = inputs @ inputs.T

In [67]:
attention_score

tensor([[ 6.8010, -5.0433,  3.0749, -2.8109, -1.3072,  0.4211],
        [-5.0433, 10.0106, -3.5345,  0.1732,  2.4690,  0.2271],
        [ 3.0749, -3.5345,  3.0423, -0.7557, -2.6776, -1.5917],
        [-2.8109,  0.1732, -0.7557, 11.4238, -1.2816, -0.7969],
        [-1.3072,  2.4690, -2.6776, -1.2816,  3.0359,  2.2199],
        [ 0.4211,  0.2271, -1.5917, -0.7969,  2.2199,  2.0822]],
       grad_fn=<MmBackward0>)

In [68]:
attention_weights = torch.softmax(attention_score, dim=-1)

In [69]:
attention_weights

tensor([[9.7451e-01, 6.9964e-06, 2.3475e-02, 6.5227e-05, 2.9340e-04, 1.6522e-03],
        [2.8966e-07, 9.9936e-01, 1.3097e-06, 5.3383e-05, 5.3021e-04, 5.6340e-05],
        [4.9914e-01, 6.7265e-04, 4.8308e-01, 1.0829e-02, 1.5846e-03, 4.6940e-03],
        [6.5757e-07, 1.2999e-05, 5.1342e-06, 9.9997e-01, 3.0346e-06, 4.9271e-06],
        [6.3735e-03, 2.7820e-01, 1.6190e-03, 6.5389e-03, 4.9040e-01, 2.1686e-01],
        [7.3743e-02, 6.0740e-02, 9.8536e-03, 2.1814e-02, 4.4559e-01, 3.8826e-01]],
       grad_fn=<SoftmaxBackward0>)

In [70]:
context_vectors = attention_weights @ inputs

In [71]:
context_vectors

tensor([[-0.6318,  2.0879, -0.2896,  1.3261],
        [ 0.9065, -0.3508, -0.6824, -2.9309],
        [-0.1403,  1.3306, -0.4815,  1.3457],
        [ 2.4882, -0.4090,  2.2378,  0.2384],
        [-0.4165, -0.0113,  0.3164, -1.5683],
        [-0.7341,  0.2758,  0.5862, -0.8721]], grad_fn=<MmBackward0>)

In [75]:
class SelfAttention(nn.Module):
    def __init__(self, context_length, dim_in):
        super().__init__()
        self.context_length = context_length
        self.dim_in = dim_in

    def forward(self, x):
        attention_scores = x @ x.T
        attention_weights = torch.softmax(attention_scores, dim=-1)
        context_vectors = attention_weights @ x
        return context_vectors

In [76]:
simple_attention = SelfAttention(6, emb_dim)

In [74]:
simple_attention(inputs)

tensor([[-0.6318,  2.0879, -0.2896,  1.3261],
        [ 0.9065, -0.3508, -0.6824, -2.9309],
        [-0.1403,  1.3306, -0.4815,  1.3457],
        [ 2.4882, -0.4090,  2.2378,  0.2384],
        [-0.4165, -0.0113,  0.3164, -1.5683],
        [-0.7341,  0.2758,  0.5862, -0.8721]], grad_fn=<MmBackward0>)